In [6]:
import pandas as pd

In [34]:
import os
folder = r"C:\Users\ASPIRE A715 - 42G\Downloads\MentalHealth_AI_Project\data\raw\VMHQA\xlsx"
print(os.listdir(folder))

['[SHARING]_FinalData_10000_18082024.xlsx', '[SHARING]_final_full_10066.xlsx']


In [4]:
file = r"C:\Users\ASPIRE A715 - 42G\Downloads\MentalHealth_AI_Project\data\processed\train.csv"

In [8]:
# Lấy cột Topic, loại bỏ NaN, trim khoảng trắng
df = pd.read_csv(file)

print("Danh sách cột:", df.columns.tolist())

topics = df['label'].dropna().astype(str).str.strip()
topic_counts = topics.value_counts()

# Ghi ra file
output_file = "unique_topics.txt"
with open(output_file, "w", encoding="utf-8") as f:
    for topic, count in topic_counts.items():
        f.write(f"{topic}: {count}\n")

print(f"Đã lưu {len(topic_counts)} topic vào file '{output_file}'")


Danh sách cột: ['text', 'label']
Đã lưu 68 topic vào file 'unique_topics.txt'


In [40]:
import pandas as pd
import re

# Đọc danh sách topic từ file bạn đã xuất
with open("unique_topics.txt", "r", encoding="utf-8") as f:
    topics = [line.strip() for line in f if line.strip()]

# Định nghĩa các nhóm và từ khóa tương ứng (có thể mở rộng)
groups = {
    "Rối loạn tâm thần nặng": [
        "loạn thần", "tâm thần phân liệt", "hoang tưởng", "ảo thanh", "ảo giác",
        "catatonia", "rối loạn phân liệt", "hoang tưởng bị hại", "hoang tưởng được yêu"
    ],
    "Rối loạn khí sắc": [
        "trầm cảm", "hưng cảm", "lưỡng cực", "khí sắc", "cyclothymic", "rối loạn cảm xúc theo mùa",
        "trầm cảm sau sinh", "rối loạn điều hòa khí sắc"
    ],
    "Rối loạn lo âu": [
        "lo âu", "sợ", "ám ảnh sợ", "hoảng sợ", "hoảng loạn", "panic", "gad", "sợ không gian hẹp",
        "sợ khoảng trống", "sợ đám đông", "sợ độ cao", "sợ máu", "sợ bệnh viện"
    ],
    "Rối loạn ám ảnh cưỡng chế": [
        "ám ảnh cưỡng chế", "ocd", "cưỡng bức", "cưỡng chế", "ám ảnh", "tích trữ", "hoarding"
    ],
    "Rối loạn liên quan stress & sang chấn": [
        "stress", "ptsd", "sang chấn", "căng thẳng", "chấn thương", "post-traumatic",
        "rối loạn stress", "hậu chấn thương", "trauma"
    ],
    "Rối loạn ăn uống": [
        "chán ăn", "ăn uống", "bulimia", "biếng ăn", "ăn vô độ", "pica", "rối loạn ăn"
    ],
    "Rối loạn giấc ngủ": [
        "mất ngủ", "ngủ", "ác mộng", "ngưng thở", "sleep", "insomnia", "ngủ rũ", "mộng du"
    ],
    "Rối loạn phát triển": [
        "tự kỷ", "autism", "adhd", "tăng động", "chậm phát triển", "rett", "asperger",
        "học tập", "ngôn ngữ", "phát triển", "tourette", "tic"
    ],
    "Rối loạn nhân cách": [
        "nhân cách", "ái kỷ", "ranh giới", "chống đối xã hội", "kịch tính", "né tránh", "phụ thuộc",
        "hoang tưởng (nhân cách)", "phân liệt", "ám ảnh tính cách"
    ],
    "Rối loạn kiểm soát xung động": [
        "nghiện", "cá cược", "bộc phát", "giận dữ", "xung động", "kiềm chế", "cướp giật", "đốt phá", "nhổ tóc"
    ],
    "Rối loạn phân ly": [
        "phân ly", "giải thể nhân cách", "mất trí nhớ phân ly", "đa nhân cách", "dissociative"
    ],
    "Rối loạn triệu chứng cơ thể": [
        "dạng cơ thể", "triệu chứng cơ thể", "bệnh tưởng", "hypochondria", "somatic"
    ],
    "Rối loạn tình dục & bản dạng giới": [
        "tình dục", "bản dạng giới", "lưỡng tính", "phiền muộn giới tính", "rối loạn dục tính",
        "thị dâm", "phô dâm", "ái tử thi", "ái vật"
    ]
}

# Hàm gán nhóm dựa trên từ khóa
def assign_group(topic):
    topic_lower = topic.lower()
    # Ưu tiên các từ khóa đặc biệt
    for group, keywords in groups.items():
        for kw in keywords:
            if kw in topic_lower:
                return group
    # Nhóm "Khác (vấn đề tâm lý)" cho các khái niệm liên quan đến tâm lý nhưng không phải rối loạn
    psychological_concepts = [
        "tâm lý", "cảm xúc", "hành vi", "nhận thức", "tri giác", "trí nhớ", "tư duy",
        "động cơ", "nhu cầu", "giao tiếp", "xã hội", "học thuyết", "hiệu ứng", "liệu pháp",
        "trị liệu", "tham vấn", "test", "trắc nghiệm", "thang đo", "phát triển", "giáo dục",
        "stress (không bệnh)", "burnout"
    ]
    for kw in psychological_concepts:
        if kw in topic_lower:
            return "Khác (vấn đề tâm lý)"
    return "Khác (không phải rối loạn)"

# Tạo mapping
mapping = []
for topic in topics:
    group = assign_group(topic)
    mapping.append({"topic": topic, "group": group})

df_mapping = pd.DataFrame(mapping)
# Lưu file CSV
df_mapping.to_csv("topic_to_group.csv", index=False, encoding="utf-8-sig")
print(f"Đã tạo file mapping cho {len(df_mapping)} topic. Lưu tại topic_to_group.csv")

Đã tạo file mapping cho 1497 topic. Lưu tại topic_to_group.csv
